# Stellar Classification AutoGluon Baseline

这个 notebook 是第一版 Kaggle baseline：读取数据、构造少量天文表格特征、做分层验证、训练 AutoGluon、查看 leaderboard，并生成 `submission.csv`。本地推荐优先运行 `scripts/01_baseline_autogluon.py`，notebook 主要用于交互式检查。

In [ ]:
from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from autogluon.tabular import TabularPredictor

## 1. 参数

In [ ]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'autogluon_baseline'
RUN_NAME = datetime.now().strftime('%Y%m%d_%H%M%S')

ID_COL = 'id'
TARGET = 'class'
EVAL_METRIC = 'accuracy'
RANDOM_STATE = 42
VALID_SIZE = 0.2

# 首版 baseline 先拿稳定结果；有更多时间时可改成 'best_quality' 或增大 TIME_LIMIT。
PRESETS = 'medium_quality'
TIME_LIMIT = 900
SKIP_REFIT_FULL = True

run_dir = OUTPUT_DIR / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
model_dir = run_dir / 'ag_model'
run_dir

## 2. 读取数据

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train:', train.shape)
print('test:', test.shape)
print('sample_submission:', sample_submission.shape)
display(train.head())
display(train[TARGET].value_counts())
display(train.isna().sum())

## 3. 特征工程

In [ ]:
MAG_COLS = ['u', 'g', 'r', 'i', 'z']
CATEGORICAL_COLS = ['spectral_type', 'galaxy_population']

def add_astronomy_features(df):
    df = df.copy()
    if all(col in df.columns for col in MAG_COLS):
        for left, right in [('u', 'g'), ('g', 'r'), ('r', 'i'), ('i', 'z'), ('u', 'r'), ('g', 'i'), ('r', 'z'), ('u', 'z')]:
            df[f'{left}_minus_{right}'] = df[left] - df[right]
        mag_values = df[MAG_COLS]
        df['mag_mean'] = mag_values.mean(axis=1)
        df['mag_std'] = mag_values.std(axis=1)
        df['mag_min'] = mag_values.min(axis=1)
        df['mag_max'] = mag_values.max(axis=1)
        df['mag_range'] = df['mag_max'] - df['mag_min']
    if 'redshift' in df.columns and all(col in df.columns for col in MAG_COLS):
        for col in MAG_COLS:
            df[f'{col}_redshift_ratio'] = df[col] / (df['redshift'].abs() + 1e-3)
        df['redshift_x_mag_mean'] = df['redshift'] * df['mag_mean']
    if 'alpha' in df.columns:
        alpha_rad = np.deg2rad(df['alpha'])
        df['alpha_sin'] = np.sin(alpha_rad)
        df['alpha_cos'] = np.cos(alpha_rad)
    if 'delta' in df.columns:
        delta_rad = np.deg2rad(df['delta'])
        df['delta_sin'] = np.sin(delta_rad)
        df['delta_cos'] = np.cos(delta_rad)
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype('category')
    return df

train_fe = add_astronomy_features(train).drop(columns=[ID_COL])
test_fe = add_astronomy_features(test).drop(columns=[ID_COL])
print(train_fe.shape, test_fe.shape)
train_fe.head()

## 4. 分层验证

In [ ]:
train_part, valid_part = train_test_split(
    train_fe,
    test_size=VALID_SIZE,
    stratify=train_fe[TARGET],
    random_state=RANDOM_STATE,
)
train_part = train_part.reset_index(drop=True)
valid_part = valid_part.reset_index(drop=True)
print(train_part.shape, valid_part.shape)
display(valid_part[TARGET].value_counts(normalize=True))

## 5. 训练 AutoGluon

In [ ]:
predictor = TabularPredictor(
    label=TARGET,
    problem_type='multiclass',
    eval_metric=EVAL_METRIC,
    path=str(model_dir),
    verbosity=2,
)

predictor.fit(
    train_data=train_part,
    tuning_data=valid_part,
    presets=PRESETS,
    time_limit=TIME_LIMIT,
    num_gpus=0,
)

## 6. 验证与 leaderboard

In [ ]:
leaderboard = predictor.leaderboard(valid_part, silent=True)
valid_metrics = predictor.evaluate(valid_part, silent=True)

leaderboard.to_csv(run_dir / 'leaderboard_valid.csv', index=False)
with open(run_dir / 'valid_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(valid_metrics, f, indent=2, ensure_ascii=False, default=str)

display(leaderboard)
valid_metrics

## 7. 可选：全量 refit

In [ ]:
if not SKIP_REFIT_FULL:
    refit_map = predictor.refit_full(set_best_to_refit_full=True, train_data_extra=valid_part)
    with open(run_dir / 'refit_full_map.json', 'w', encoding='utf-8') as f:
        json.dump(refit_map, f, indent=2, ensure_ascii=False, default=str)
    refit_map

## 8. 生成提交文件

In [ ]:
test_pred = predictor.predict(test_fe)

submission = sample_submission.copy()
submission[ID_COL] = test[ID_COL].values
submission[TARGET] = test_pred.values

submission_path = run_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(submission_path.resolve())
display(submission.head())
display(submission[TARGET].value_counts())

## 9. 下一步

- 提交首版 `submission.csv`，记录 public score。
- 对比 `medium_quality`、`high_quality`、`best_quality` 和不同 `time_limit`。
- 开启 `SKIP_REFIT_FULL = False` 做全量重训。
- 检查各类别召回率，特别是 `STAR` 和 `QSO` 的混淆。